# 45 — Robustness checks (regression tests, isolated from operational code)

Adversarial fixtures and assertions that none of the following may reintroduce retired content. These are **test inputs only** — the operational notebooks never generate files. The four adversarial files are written into `landing/` for the test batch and **removed automatically when the run finishes** (`cleanup_robustness_fixtures()`), so a normal ingest pass after testing never encounters intentionally invalid or conflicting files. Legitimate source files and accepted history (including the inactive future-dated version) are preserved; quarantine observations remain in the table as historical records (deduplicated on re-runs).

When run standalone, this notebook `%run`s `40_demo_generation` first, which is idempotent and leaves the completed V2 → V3 demo state (AGR-001 on V3). All checks below assume that state.

The future-dated fixture uses a business date **relative to the test run** (current date + 30 days), so the future-inactive check never expires.

In [0]:
%run ./40_demo_generation

In [0]:
# Robustness tests: none of the following may reintroduce retired content.
#
# Expected results, all asserted below:
#   - AGR-001 stays on V3; the late-arriving V1 file is accepted as history but
#     never becomes active (date-based eligibility, not arrival order).
#   - broken_metadata.md is quarantined with a parse reason; conflicting_v3.md
#     is quarantined for conflicting content — and its same-key sibling
#     new_field_manual_1.md is quarantined too, because the preflight rejects
#     the whole ambiguous (document_id, version_id) batch while accepted V3
#     history stays intact and active.
#   - future_dated.md (AGR-006, run date + 30 days) is stored as history but inactive.
#   - Stored embeddings are reused: chunk_embeddings row count does not grow
#     for an already-embedded active pool (this shows no NEW vectors were
#     persisted; it does not by itself count model inference calls).
#   - Re-ingesting unchanged files stays idempotent; the grounded answer still
#     cites only AGR-001-V3 with the 6-day interval.
#
#   - After the assertions, the four adversarial fixture files are removed from
#     landing/ (cleanup_robustness_fixtures), so the next normal ingest pass
#     reports zero quarantined files.
#
# Side effects: quarantine observations stay in the table as historical records
# (deduplicated on re-runs); the fixture files themselves are cleaned up.
q = 'What is the irrigation review interval for wheat?'
emb_before = spark.table(table('chunk_embeddings')).count()
baseline_answer = ask(q, 'AGR-001')
emb_after = spark.table(table('chunk_embeddings')).count()
assert emb_after == emb_before, 'Stored embedding rows grew for an already-embedded active pool'
assert baseline_answer['mode'] == 'GROUNDED_LLM' and '6 days' in baseline_answer['answer']
print(f'PASS: no additional embedding vectors stored ({emb_before} rows unchanged; missing-vector reuse — row counts do not count model inference calls).')

# 1) Late-arriving OLD file (V1 content, new filename) must not reactivate retired content.
(LANDING / 'late_arrival_legacy_1.md').write_text(make_document(1, 1))
# 2) Invalid metadata (missing required fields) must be quarantined.
(LANDING / 'broken_metadata.md').write_text('---metadata\n{"document_id": "AGR-001"}\n---\nbody without required fields')
# 3) Conflicting content for an existing (document_id, version_id) must be quarantined.
(LANDING / 'conflicting_v3.md').write_text(make_document(1, 3).replace('is 6 days', 'is 4 days'))
# 4) Future-dated version is stored in history but never becomes active.
future_date = (date.today() + timedelta(days=30)).isoformat()  # relative to the test run
future = (make_document(1, 3)
          .replace('"AGR-001"', '"AGR-006"')
          .replace('AGR-001-V3', 'AGR-006-V1')
          .replace('2026-07-15', future_date))
(LANDING / 'future_dated.md').write_text(future)
print('Future-dated fixture business date:', future_date)

result = ingest_and_publish()
print(result)

registry = registry_snapshot()
assert registry['AGR-001'][0] == 'AGR-001-V3', 'Late-arriving old file changed the active version'
assert 'AGR-006' not in registry, 'Future-dated version became active'
active_texts = [r.chunk_text for r in spark.table(table('active_document_chunks'))
                .filter(F.col('document_id') == 'AGR-001').collect()]
assert active_texts, 'No active chunks for AGR-001'
for retired in ('LEGACY-RED', '10 days', '8 days', '4 days'):
    assert not any(retired in t for t in active_texts), f'Retired content leaked: {retired}'

reasons = {r.source_path.split('/')[-1]: r.reason
           for r in spark.table(table('quarantined_documents')).collect()}
assert 'Missing or invalid' in reasons.get('broken_metadata.md', ''), 'Invalid metadata not quarantined'
assert reasons.get('conflicting_v3.md') == 'Conflicting version content', 'Conflicting content not quarantined'
# An ambiguous batch also quarantines the sibling file claiming the same version key:
# the preflight rejects the whole (document_id, version_id) batch, not just one file.
assert reasons.get('new_field_manual_1.md') == 'Conflicting version content', 'Ambiguous batch not fully rejected'

future_row = spark.table(table('document_manifest')).filter(F.col('document_id') == 'AGR-006').first()
assert future_row is not None and future_row.is_active is False, 'Future version missing or wrongly active'

counts_before = (spark.table(table('document_versions')).count(), spark.table(table('document_chunks')).count())
ingest_and_publish()
assert counts_before == (spark.table(table('document_versions')).count(), spark.table(table('document_chunks')).count())

final_answer = ask(q, 'AGR-001')
assert final_answer['mode'] == 'GROUNDED_LLM'
assert all(s['version_id'] == 'AGR-001-V3' for s in final_answer['sources'])
assert '6 days' in final_answer['answer'] and '8 days' not in final_answer['answer']
print('PASS: late-arriving old files, invalid metadata, conflicting content and future dates cannot reintroduce retired content.')
print('PASS: grounded answer still cites only V3 with the 6-day interval.')

# --- Fixture cleanup: keep the operational landing directory clean ----------------
def cleanup_robustness_fixtures():
    """Remove THIS notebook's adversarial fixture files from landing/ (idempotent).

    Deletes only the four files written by the pass above. Legitimate source
    files, accepted history (including the inactive future-dated version), and
    the quarantine observations are preserved, so the next normal ingest pass
    encounters no intentionally invalid or conflicting files.
    """
    removed = []
    for name in ('late_arrival_legacy_1.md', 'broken_metadata.md',
                 'conflicting_v3.md', 'future_dated.md'):
        path = LANDING / name
        if path.exists():
            path.unlink()
            removed.append(name)
    return removed

removed = cleanup_robustness_fixtures()
print('Robustness fixtures removed from landing/:', removed or 'none present')
remaining = {p.name for p in LANDING.iterdir() if p.suffix.lower() in ('.md', '.txt')}
assert not remaining & {'late_arrival_legacy_1.md', 'broken_metadata.md',
                        'conflicting_v3.md', 'future_dated.md'}, 'Fixture files still present'
post_cleanup = ingest_and_publish()
print('Post-cleanup ingest:', post_cleanup)
assert post_cleanup['quarantined_files'] == 0, 'Operational landing still contains intentional robustness fixtures'
print('PASS: operational landing is free of intentional robustness fixtures.')